### transformations

In [ ]:
# -------------------------------------------
# PySpark RDD Transformations - Example Code
# -------------------------------------------

from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("RDD_Transformations_Examples").getOrCreate()

# Create a simple RDD
rdd = spark.sparkContext.parallelize([1, 2, 3, 4, 5])

# ---------------------------
# 1. map(func)
# Apply a function to each element
mapped_rdd = rdd.map(lambda x: x * 2)   # [2, 4, 6, 8, 10]

# ---------------------------
# 2. filter(func)
# Keep only elements that satisfy condition
filtered_rdd = rdd.filter(lambda x: x % 2 == 0)   # [2, 4]

# ---------------------------
# 3. flatMap(func)
# Each element can map to multiple output elements
rdd2 = spark.sparkContext.parallelize(["hello world", "hi all"])
flatmapped_rdd = rdd2.flatMap(lambda x: x.split(" "))  # ["hello", "world", "hi", "all"]

# ---------------------------
# 4. mapPartitions(func)
# Apply function on each partition
def add_one(iterator):
    return (x + 1 for x in iterator)

map_partition_rdd = rdd.mapPartitions(add_one)  # Each element +1

# ---------------------------
# 5. mapPartitionsWithIndex(func)
def show_partition(index, iterator):
    return (f"Partition {index}: {x}" for x in iterator)

map_part_index_rdd = rdd.mapPartitionsWithIndex(show_partition)

# ---------------------------
# 6. sample(withReplacement, fraction, seed)
sample_rdd = rdd.sample(False, 0.4, 1)

# ---------------------------
# 7. union(otherDataset)
union_rdd = rdd.union(spark.sparkContext.parallelize([6, 7]))

# ---------------------------
# 8. intersection(otherDataset)
intersect_rdd = rdd.intersection(spark.sparkContext.parallelize([3, 4, 9]))  # [3, 4]

# ---------------------------
# 9. distinct()
distinct_rdd = spark.sparkContext.parallelize([1, 1, 2, 2, 3]).distinct()  # [1,2,3]

# ---------------------------
# Key-Value RDD for next examples
kv_rdd = spark.sparkContext.parallelize([("a", 1), ("b", 2), ("a", 3), ("b", 4)])

# ---------------------------
# 10. groupByKey()
grouped_rdd = kv_rdd.groupByKey()   # ("a", [1,3]), ("b", [2,4])

# ---------------------------
# 11. reduceByKey(func)
reduced_rdd = kv_rdd.reduceByKey(lambda x, y: x + y)   # ("a",4), ("b",6)

# ---------------------------
# 12. aggregateByKey()
# zeroValue = 0, seqOp = sum values inside partition, combOp = sum across partitions
aggregated_rdd = kv_rdd.aggregateByKey(0, lambda x, y: x + y, lambda x, y: x + y)

# ---------------------------
# 13. sortByKey()
sorted_rdd = kv_rdd.sortByKey(ascending=True)

# ---------------------------
# 14. join(otherDataset)
other_kv_rdd = spark.sparkContext.parallelize([("a", 20), ("b", 40)])
joined_rdd = kv_rdd.join(other_kv_rdd)  # ("a",(1,20)), ("a",(3,20)), ("b",(2,40)), ("b",(4,40))

# ---------------------------
# 15. cogroup(otherDataset)
cogroup_rdd = kv_rdd.cogroup(other_kv_rdd)
# ("a", ([1,3], [20])), ("b", ([2,4], [40]))

# ---------------------------
# 16. cartesian(otherDataset)
cartesian_rdd = rdd.cartesian(spark.sparkContext.parallelize([10, 20]))
# (1,10),(1,20),(2,10)...

# ---------------------------
# 17. coalesce(numPartitions)
coalesced_rdd = rdd.coalesce(2)

# ---------------------------
# 18. repartition(numPartitions)
repartitioned_rdd = rdd.repartition(4)

# ---------------------------
# 19. repartitionAndSortWithinPartitions(partitioner)
from pyspark.rdd import portable_hash
partitioned_sorted_rdd = kv_rdd.repartitionAndSortWithinPartitions(
    numPartitions=2,
    partitionFunc=lambda key: portable_hash(key)
)

# ---------------------------
# Print example outputs
print("map:", mapped_rdd.collect())
print("filter:", filtered_rdd.collect())
print("flatMap:", flatmapped_rdd.collect())
print("mapPartitions:", map_partition_rdd.collect())
print("mapPartitionsWithIndex:", map_part_index_rdd.collect())
print("sample:", sample_rdd.collect())
print("union:", union_rdd.collect())
print("intersection:", intersect_rdd.collect())
print("distinct:", distinct_rdd.collect())
print("groupByKey:", [(k, list(v)) for k,v in grouped_rdd.collect()])
print("reduceByKey:", reduced_rdd.collect())
print("aggregateByKey:", aggregated_rdd.collect())
print("sortByKey:", sorted_rdd.collect())
print("join:", joined_rdd.collect())
print("cogroup:", [(k, (list(v1), list(v2))) for k,(v1,v2) in cogroup_rdd.collect()])
print("cartesian:", cartesian_rdd.collect())
print("Done!")

### actions

In [ ]:
# -------------------------------------------
# RDD ACTIONS EXAMPLE IN Pyspark
# -------------------------------------------
# This script demonstrates common RDD *Actions* in PySpark
# Each example is commented simply and clearly.

from pyspark import SparkContext

# creating spark context
sc = SparkContext("local", "ActionsDemo")

# our sample dataset
data = [5, 3, 10, 3, 7, 3, 10, 1]

# converting python list into RDD
rdd = sc.parallelize(data)

# --------------------------------------------------------
# 1. reduce(func)
# applies a function to combine all elements
# here: adding all numbers to get their sum
result_reduce = rdd.reduce(lambda a, b: a + b)
print("reduce(): sum of all elements =", result_reduce)

# --------------------------------------------------------
# 2. collect()
# brings all RDD data to the driver (only use for small datasets)
result_collect = rdd.collect()
print("collect():", result_collect)

# --------------------------------------------------------
# 3. count()
# number of elements in the RDD
result_count = rdd.count()
print("count():", result_count)

# --------------------------------------------------------
# 4. first()
# returns the first element
result_first = rdd.first()
print("first():", result_first)

# --------------------------------------------------------
# 5. take(n)
# returns first 'n' elements as a list
result_take = rdd.take(3)
print("take(3):", result_take)

# --------------------------------------------------------
# 6. takeSample(withReplacement, num, seed)
# here we take 4 elements randomly without replacement
result_sample = rdd.takeSample(False, 4, 42)
print("takeSample():", result_sample)

# --------------------------------------------------------
# 7. takeOrdered(n)
# returns smallest 'n' elements by default
result_ordered = rdd.takeOrdered(3)
print("takeOrdered(3):", result_ordered)

# --------------------------------------------------------
# 8. saveAsTextFile(path)
# saves RDD to text file storage (local/HDFS/etc.)
# NOTE: Uncomment to use
# rdd.saveAsTextFile("output_text_file")

# --------------------------------------------------------
# 9. countByKey()
# only works for (K, V) RDDs
pair_rdd = sc.parallelize([("apple", 1), ("banana", 1), ("apple", 1), ("orange", 1)])
result_countByKey = pair_rdd.countByKey()
print("countByKey():", result_countByKey)

# --------------------------------------------------------
# 10. foreach(func)
# runs a function for every element (usually used for side-effects)
print("foreach(): printing each element:")
rdd.foreach(lambda x: print(x))

# --------------------------------------------------------
print("\nAll actions executed successfully.")
sc.stop()